# 06 — Final Labeler Selection, Validation Audit, and Full-Corpus Re-score Plan

This notebook is the decision point for the project’s two text-based measures:

- **AI exposure:** does the role itself involve using/building/working with AI or ML?
- **Automatability:** are the role’s core tasks routine enough that current computers/AI could perform most of them?

Important terminology: `data/processed/gold_labels.csv` is kept as a filename for compatibility, but the labels are treated as a **100-row Claude-drafted validation audit set**, not as independent human-reviewed annotation. The metrics below are useful for model selection and face-validity checks, but exact AUC/F1 values are uncertain because the audit set is small, especially for exposure (`5` positives).

The notebook has three jobs:

1. Pick the best labeler from the cached validation-audit evidence.
2. Report uncertainty and keep weaker models in a compact robustness appendix.
3. Provide a gated full-corpus re-score cell for the chosen final model(s).


## Setup

`FULL_RESCORE` is off by default. All validation/model-selection cells use cached files and spend no API calls. Turn `FULL_RESCORE=True` only after you have a working API key and model endpoint.


In [41]:
import os, re, json, time, math, copy, warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.base import clone

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
PRIMARY_FULL_CORPUS_MODEL = "gpt-5.4-mini"  # existing full-corpus artifact currently on disk
FULL_RESCORE = True                         # flip only when API credentials are ready
MAX_CHARS = 1500

HERE = Path.cwd()
ROOT = next((pp for pp in [HERE, *HERE.parents] if (pp / "data" / "processed").exists()), HERE)
PROC = ROOT / "data" / "processed"
SWEEP_DIR = PROC / "sweep"
FIG_DIR = ROOT / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("project root:", ROOT)
print("processed dir:", PROC)
print("FULL_RESCORE:", FULL_RESCORE)


project root: /Users/PeakViprakasit/QSS45_Final_Project
processed dir: /Users/PeakViprakasit/QSS45_Final_Project/data/processed
FULL_RESCORE: True


In [42]:
import os, getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")


OpenAI API key:  ········
Anthropic API key:  ········


## Load Validation Audit and Full Corpus

Diagnostics are printed before model comparison so file/merge problems are visible, which matches the final-project repo rubric.


In [43]:
validation = pd.read_csv(PROC / "gold_labels.csv", dtype={"job_id": str})
postings = pd.read_csv(PROC / "postings_clean.csv", dtype={"job_id": str})
bert_full = pd.read_csv(PROC / "zeroshot_scores.csv", dtype={"job_id": str})

print("validation-audit rows:", len(validation))
print("validation labeler values:", validation.get("labeler", pd.Series(dtype=str)).value_counts(dropna=False).to_dict())
print("exposure positives:", int(validation.label_exposure.sum()))
print("automatability positives:", int(validation.label_automatable.sum()))
print("full corpus rows:", len(postings), "| unique job_id:", postings.job_id.nunique())
print("BERT full rows:", len(bert_full))

# Add full text to validation rows. The labels file stores excerpts; model re-scoring uses cleaned full descriptions.
validation = validation.merge(
    postings[["job_id", "description_clean"]], on="job_id", how="left", suffixes=("", "_full")
)
validation["trunc_text"] = validation["description_clean"].fillna("").astype(str).str[:MAX_CHARS]
assert validation["trunc_text"].str.len().gt(0).all(), "Some validation rows are missing posting text."


validation-audit rows: 100
validation labeler values: {'claude-draft': 100}
exposure positives: 5
automatability positives: 18
full corpus rows: 12487 | unique job_id: 12487
BERT full rows: 3414


## Metrics With Uncertainty

With only 5 exposure positives and 18 automatability positives, point estimates can swing. The bootstrap intervals below resample the 100 validation-audit rows and show how much uncertainty sits around AUC/F1.


In [44]:
def metrics_block(y, score, thr=0.5):
    y = np.asarray(y, float)
    score = np.asarray(score, float)
    ok = ~np.isnan(score)
    y, score = y[ok].astype(int), score[ok]
    pred = (score >= thr).astype(int)
    auc = roc_auc_score(y, score) if len(set(y)) > 1 else np.nan
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
    caught = int(((pred == 1) & (y == 1)).sum())
    return dict(n=len(y), pos=int(y.sum()), AUC=auc, precision=p, recall=r, F1=f,
                acc=accuracy_score(y, pred), caught=f"{caught}/{int(y.sum())}")


def bootstrap_ci(y, score, metric="F1", thr=0.5, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    y = np.asarray(y).astype(int)
    score = np.asarray(score).astype(float)
    vals = []
    n = len(y)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        # Skip degenerate resamples for AUC.
        if metric == "AUC" and len(np.unique(y[idx])) < 2:
            continue
        vals.append(metrics_block(y[idx], score[idx], thr=thr)[metric])
    if not vals:
        return (np.nan, np.nan)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return round(float(lo), 3), round(float(hi), 3)


def fmt_metric_dict(d):
    out = {}
    for k, v in d.items():
        out[k] = round(v, 3) if isinstance(v, (float, np.floating)) else v
    return out


## Cached Labeler Candidates

In [45]:
def _ok_mask(df):
    if "raw_ok" not in df.columns:
        return pd.Series(True, index=df.index)
    return df["raw_ok"].fillna(0).astype(str).isin(["1", "1.0", "True", "true"])


def load_sweep_candidates(validation_ids):
    candidates, inventory = {}, []
    for path in sorted(SWEEP_DIR.glob("*.csv")):
        raw = pd.read_csv(path, dtype={"job_id": str})
        ok = _ok_mask(raw)
        cols = ["job_id", "p_exposure", "p_automatable"]
        if "rationale" in raw.columns:
            cols.append("rationale")
        merged = pd.DataFrame({"job_id": validation_ids}).merge(raw.loc[ok, cols], on="job_id", how="left")
        complete = merged[["p_exposure", "p_automatable"]].notna().all().all()
        inventory.append({"candidate": path.stem, "rows": len(raw), "parsed_ok": int(ok.sum()),
                          "complete_validation_cache": bool(complete)})
        if complete:
            candidates[path.stem] = merged
    return candidates, pd.DataFrame(inventory)

candidates, sweep_inventory = load_sweep_candidates(validation["job_id"])
print("complete cached candidates:", len(candidates))
sweep_inventory.sort_values(["complete_validation_cache", "parsed_ok", "candidate"], ascending=[False, False, True])


complete cached candidates: 8


,candidate,rows,parsed_ok,complete_validation_cache
0,anthropic.claude-haiku-4-5-20251001,100,100,True
6,claude-haiku-4-5,100,100,True
7,claude-opus-4-6,100,100,True
8,claude-opus-4-8,100,100,True
9,claude-sonnet-4-6,100,100,True
11,gpt-5.4,100,100,True
10,gpt-5.4-mini,100,100,True
14,openai.gpt-oss-120b,100,100,True
2,anthropic.claude-opus-4-6,100,93,False
13,openai.gpt-5.5-2026-04-23,100,57,False


In [46]:
rows = []
rows.append(dict(construct="exposure", model="BERT", family="baseline",
                 **metrics_block(validation.label_exposure, validation.bert_exposure_score)))
rows.append(dict(construct="automatability", model="BERT", family="baseline",
                 **metrics_block(validation.label_automatable, validation.bert_replace_score)))

for name, pred in candidates.items():
    rows.append(dict(construct="exposure", model=name, family="zero-shot LLM",
                     **metrics_block(validation.label_exposure, pred.p_exposure)))
    rows.append(dict(construct="automatability", model=name, family="zero-shot LLM",
                     **metrics_block(validation.label_automatable, pred.p_automatable)))

candidate_metrics = pd.DataFrame(rows)
for metric in ["AUC", "precision", "recall", "F1", "acc"]:
    candidate_metrics[metric] = candidate_metrics[metric].round(3)

candidate_metrics.sort_values(["construct", "F1", "AUC"], ascending=[True, False, False]).reset_index(drop=True)


,construct,model,family,n,pos,AUC,precision,recall,F1,acc,caught
0,automatability,claude-opus-4-8,zero-shot LLM,100,18,0.923,0.625,0.833,0.714,0.88,15/18
1,automatability,gpt-5.4-mini,zero-shot LLM,100,18,0.871,0.484,0.833,0.612,0.81,15/18
2,automatability,gpt-5.4,zero-shot LLM,100,18,0.893,0.556,0.556,0.556,0.84,10/18
3,automatability,claude-sonnet-4-6,zero-shot LLM,100,18,0.922,0.857,0.333,0.480,0.87,6/18
4,automatability,anthropic.claude-haiku-4-5-20251001,zero-shot LLM,100,18,0.818,0.444,0.444,0.444,0.80,8/18
5,automatability,claude-haiku-4-5,zero-shot LLM,100,18,0.848,0.421,0.444,0.432,0.79,8/18
6,automatability,openai.gpt-oss-120b,zero-shot LLM,100,18,0.835,0.600,0.333,0.429,0.84,6/18
7,automatability,claude-opus-4-6,zero-shot LLM,100,18,0.903,0.833,0.278,0.417,0.86,5/18
8,automatability,BERT,baseline,100,18,0.605,1.000,0.056,0.105,0.83,1/18
9,exposure,gpt-5.4,zero-shot LLM,100,5,0.972,1.000,0.800,0.889,0.99,4/5


## Serious Candidate Set

The full leaderboard is useful for the appendix, but the main decision should focus on plausible project choices:

- BERT baseline.
- Existing implemented full-corpus GPT artifact (`gpt-5.4-mini`).
- Best single cached LLM per construct.
- Best cached ensemble per construct.
- Best trained supervised course-style model as a robustness benchmark.


In [47]:
def ensemble_score(names, score_col, reducer="mean"):
    arr = np.vstack([candidates[n][score_col].astype(float).values for n in names])
    return np.median(arr, axis=0) if reducer == "median" else np.mean(arr, axis=0)

complete_names = list(candidates)
ensemble_specs = [
    ("mean_all_complete_llm", complete_names, "mean"),
    ("median_all_complete_llm", complete_names, "median"),
    ("mean_gpt_family", [n for n in complete_names if n.startswith("gpt") or n.startswith("openai.gpt")], "mean"),
]
ens_rows = []
for ens_name, names, reducer in ensemble_specs:
    if not names:
        continue
    ens_rows.append(dict(construct="exposure", model=ens_name, family="ensemble",
                         **metrics_block(validation.label_exposure, ensemble_score(names, "p_exposure", reducer))))
    ens_rows.append(dict(construct="automatability", model=ens_name, family="ensemble",
                         **metrics_block(validation.label_automatable, ensemble_score(names, "p_automatable", reducer))))
ensemble_metrics = pd.DataFrame(ens_rows)
for metric in ["AUC", "precision", "recall", "F1", "acc"]:
    ensemble_metrics[metric] = ensemble_metrics[metric].round(3)
ensemble_metrics.sort_values(["construct", "F1", "AUC"], ascending=[True, False, False]).reset_index(drop=True)


,construct,model,family,n,pos,AUC,precision,recall,F1,acc,caught
0,automatability,mean_gpt_family,ensemble,100,18,0.899,0.522,0.667,0.585,0.83,12/18
1,automatability,mean_all_complete_llm,ensemble,100,18,0.903,0.533,0.444,0.485,0.83,8/18
2,automatability,median_all_complete_llm,ensemble,100,18,0.896,0.471,0.444,0.457,0.81,8/18
3,exposure,median_all_complete_llm,ensemble,100,5,0.983,1.000,0.800,0.889,0.99,4/5
4,exposure,mean_all_complete_llm,ensemble,100,5,0.975,1.000,0.800,0.889,0.99,4/5
5,exposure,mean_gpt_family,ensemble,100,5,0.955,1.000,0.600,0.750,0.98,3/5


## Course-Style Supervised Model Robustness

This is a real trained model, but it is trained on only 100 Claude-drafted validation-audit rows. It is useful as a class-method robustness check, not automatically the best final measurement model.


In [48]:
# Build full feature table using artifacts already on disk.
gpt_full_path = PROC / "gpt_scores.csv"
if gpt_full_path.exists():
    gpt_full = pd.read_csv(gpt_full_path, dtype={"job_id": str})[["job_id", "gpt_exposure_score", "gpt_replace_score"]]
else:
    gpt_full = pd.DataFrame({"job_id": postings.job_id, "gpt_exposure_score": np.nan, "gpt_replace_score": np.nan})

features_full = (postings
    .merge(bert_full, on="job_id", how="left")
    .merge(gpt_full, on="job_id", how="left"))

# For validation-audit rows, use exact cached primary-model sweep scores if available.
if PRIMARY_FULL_CORPUS_MODEL in candidates:
    primary_sweep = candidates[PRIMARY_FULL_CORPUS_MODEL][["job_id", "p_exposure", "p_automatable"]].rename(
        columns={"p_exposure": "gpt_exposure_sweep", "p_automatable": "gpt_replace_sweep"})
    features_full = features_full.merge(primary_sweep, on="job_id", how="left")
    features_full["gpt_exposure_score"] = features_full["gpt_exposure_sweep"].combine_first(features_full["gpt_exposure_score"])
    features_full["gpt_replace_score"] = features_full["gpt_replace_sweep"].combine_first(features_full["gpt_replace_score"])

features_full["model_text"] = (features_full["title_clean"].fillna("") + " " +
                               features_full["description_clean"].fillna("").str[:2500])
validation_model = validation[["job_id", "label_exposure", "label_automatable"]].merge(features_full, on="job_id", how="left")

flag_cols = [c for c in features_full.columns if c.startswith("flag_")]
num_cols = ["bert_exposure_score", "bert_replace_score", "gpt_exposure_score", "gpt_replace_score",
            "ai_term_count", "ai_term_freq", "word_count", "seniority_ordinal"] + flag_cols
num_cols = [c for c in num_cols if c in features_full.columns]
cat_cols = [c for c in ["industry_key", "source_platform", "employment_type_clean"] if c in features_full.columns]

pre_tabular = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
], remainder="drop")

pre_text = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ("txt", TfidfVectorizer(max_features=800, ngram_range=(1, 2), min_df=2,
                            stop_words="english", sublinear_tf=True), "model_text"),
], remainder="drop")

supervised_candidates = {
    "Unit1_score_logit": Pipeline([("pre", pre_tabular),
        ("clf", LogisticRegression(C=0.25, class_weight="balanced", solver="liblinear", random_state=42, max_iter=3000))]),
    "Unit8_tfidf_logit": Pipeline([("pre", pre_text),
        ("clf", LogisticRegression(C=0.25, class_weight="balanced", solver="liblinear", random_state=42, max_iter=3000))]),
    "Unit4_random_forest": Pipeline([("pre", pre_tabular),
        ("clf", RandomForestClassifier(n_estimators=300, max_depth=4, class_weight="balanced", random_state=42))]),
    "Unit5_gradient_boosting": Pipeline([("pre", pre_tabular),
        ("clf", GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, max_depth=2, random_state=42))]),
}

print("validation-audit rows for supervised robustness:", len(validation_model))
print("full rows available for supervised scoring:", len(features_full))
print("existing GPT full rows:", len(gpt_full), "| missing GPT scores in full feature table:", int(features_full.gpt_exposure_score.isna().sum()))


validation-audit rows for supervised robustness: 100
full rows available for supervised scoring: 12487
existing GPT full rows: 3414 | missing GPT scores in full feature table: 9073


In [49]:
def oof_scores(model, X, y):
    y = np.asarray(y).astype(int)
    n_splits = min(5, int(y.sum()), int((1 - y).sum()))
    if n_splits < 2:
        raise ValueError("Need at least two positives and two negatives for CV.")
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    out = np.zeros(len(y), dtype=float)
    for tr, te in cv.split(X, y):
        m = clone(model)
        m.fit(X.iloc[tr], y[tr])
        out[te] = m.predict_proba(X.iloc[te])[:, 1]
    return out

supervised_rows, supervised_oof = [], {}
for construct, label_col in [("exposure", "label_exposure"), ("automatability", "label_automatable")]:
    y = validation_model[label_col].astype(int).values
    for model_name, model in supervised_candidates.items():
        score = oof_scores(model, validation_model, y)
        supervised_oof[(construct, model_name)] = score
        supervised_rows.append(dict(construct=construct, model=model_name, family="trained supervised",
                                    **metrics_block(y, score)))

supervised_metrics = pd.DataFrame(supervised_rows)
for metric in ["AUC", "precision", "recall", "F1", "acc"]:
    supervised_metrics[metric] = supervised_metrics[metric].round(3)
supervised_metrics.sort_values(["construct", "F1", "AUC"], ascending=[True, False, False]).reset_index(drop=True)


,construct,model,family,n,pos,AUC,precision,recall,F1,acc,caught
0,automatability,Unit8_tfidf_logit,trained supervised,100,18,0.854,0.452,0.778,0.571,0.79,14/18
1,automatability,Unit1_score_logit,trained supervised,100,18,0.852,0.452,0.778,0.571,0.79,14/18
2,automatability,Unit4_random_forest,trained supervised,100,18,0.834,0.643,0.500,0.562,0.86,9/18
3,automatability,Unit5_gradient_boosting,trained supervised,100,18,0.769,0.500,0.333,0.400,0.82,6/18
4,exposure,Unit1_score_logit,trained supervised,100,5,0.893,0.375,0.600,0.462,0.93,3/5
5,exposure,Unit8_tfidf_logit,trained supervised,100,5,0.891,0.375,0.600,0.462,0.93,3/5
6,exposure,Unit4_random_forest,trained supervised,100,5,0.958,1.000,0.200,0.333,0.96,1/5
7,exposure,Unit5_gradient_boosting,trained supervised,100,5,0.719,0.500,0.200,0.286,0.95,1/5


## Main Model Decision

The final project should center this table, not every failed model. The weaker models still matter, but they belong in the robustness appendix below.


In [ ]:
all_models = pd.concat([candidate_metrics, ensemble_metrics, supervised_metrics], ignore_index=True)

best_single = (candidate_metrics[candidate_metrics.family == "zero-shot LLM"]
    .sort_values(["construct", "F1", "AUC", "acc"], ascending=[True, False, False, False])
    .groupby("construct", as_index=False).head(1))
best_ensemble = (ensemble_metrics
    .sort_values(["construct", "F1", "AUC", "acc"], ascending=[True, False, False, False])
    .groupby("construct", as_index=False).head(1))
best_supervised = (supervised_metrics
    .sort_values(["construct", "F1", "AUC", "acc"], ascending=[True, False, False, False])
    .groupby("construct", as_index=False).head(1))

serious_names = set(["BERT", PRIMARY_FULL_CORPUS_MODEL])
serious_names.update(best_single.model.tolist())
serious_names.update(best_ensemble.model.tolist())
serious_names.update(best_supervised.model.tolist())

main_decision = all_models[all_models.model.isin(serious_names)].copy()
main_decision["role"] = "comparison"
main_decision.loc[main_decision.model.eq("BERT"), "role"] = "free BERT baseline"
main_decision.loc[main_decision.model.eq(PRIMARY_FULL_CORPUS_MODEL), "role"] = "currently implemented full-corpus GPT"
for _, r in best_single.iterrows():
    main_decision.loc[(main_decision.construct.eq(r.construct)) & (main_decision.model.eq(r.model)), "role"] = "best single cached LLM"
for _, r in best_ensemble.iterrows():
    main_decision.loc[(main_decision.construct.eq(r.construct)) & (main_decision.model.eq(r.model)), "role"] = "best cached ensemble"
for _, r in best_supervised.iterrows():
    main_decision.loc[(main_decision.construct.eq(r.construct)) & (main_decision.model.eq(r.model)), "role"] = "best trained supervised robustness"

# Add bootstrap uncertainty for the serious candidates where scores can be reconstructed.
def get_score(row):
    model = row.model
    construct = row.construct
    if model == "BERT":
        return validation.bert_exposure_score.values if construct == "exposure" else validation.bert_replace_score.values
    if model in candidates:
        return candidates[model].p_exposure.values if construct == "exposure" else candidates[model].p_automatable.values
    if model in ensemble_metrics.model.values:
        reducer = "median" if model.startswith("median") else "mean"
        if model == "mean_gpt_family":
            names = [n for n in complete_names if n.startswith("gpt") or n.startswith("openai.gpt")]
        else:
            names = complete_names
        return ensemble_score(names, "p_exposure" if construct == "exposure" else "p_automatable", reducer)
    if (construct, model) in supervised_oof:
        return supervised_oof[(construct, model)]
    return np.repeat(np.nan, len(validation))

ci_rows = []
for _, row in main_decision.iterrows():
    y = validation.label_exposure.values if row.construct == "exposure" else validation.label_automatable.values
    s = get_score(row)
    ci_rows.append({
        "F1_95CI": bootstrap_ci(y, s, "F1"),
        "AUC_95CI": bootstrap_ci(y, s, "AUC"),
    })
main_decision = pd.concat([main_decision.reset_index(drop=True), pd.DataFrame(ci_rows)], axis=1)
main_decision = main_decision.sort_values(["construct", "F1", "AUC"], ascending=[True, False, False]).reset_index(drop=True)
main_decision


In [ ]:
# Final validation-audit winner. For implementation, prefer the best SINGLE model per construct
# because ensembles require re-scoring multiple LLMs across the full corpus.
FINAL_EXPOSURE_MODEL = best_single.loc[best_single.construct == "exposure", "model"].iloc[0]
AUDIT_BEST_AUTOMATABILITY_MODEL = best_single.loc[best_single.construct == "automatability", "model"].iloc[0]
FINAL_AUTOMATABILITY_MODEL = "gpt-5.4-mini"  # implementation choice: valid, faster/cheaper, already full-corpus proven

print("Recommended final single-model implementation:")
print("  exposure:", FINAL_EXPOSURE_MODEL)
print("  automatability:", FINAL_AUTOMATABILITY_MODEL, "(audit-best was", AUDIT_BEST_AUTOMATABILITY_MODEL + ")")
print("\nCurrent implemented full-corpus artifact:", PRIMARY_FULL_CORPUS_MODEL)
print("Existing gpt_scores.csv rows:", len(gpt_full), "out of", len(postings))

pd.DataFrame([
    {"construct": "exposure", "recommended_model": FINAL_EXPOSURE_MODEL,
     "current_full_corpus_model": PRIMARY_FULL_CORPUS_MODEL,
     "needs_full_rescore": (FINAL_EXPOSURE_MODEL != PRIMARY_FULL_CORPUS_MODEL) or (len(gpt_full) < len(postings))},
    {"construct": "automatability", "recommended_model": FINAL_AUTOMATABILITY_MODEL,
     "current_full_corpus_model": PRIMARY_FULL_CORPUS_MODEL,
     "needs_full_rescore": (FINAL_AUTOMATABILITY_MODEL != PRIMARY_FULL_CORPUS_MODEL) or (len(gpt_full) < len(postings))},
])


## Stratified Re-score With Final Model(s)

This is the implementation/robustness step. For the project, it defaults to a stratified sample rather than the entire corpus, because a full 12,487-row re-score is slow and costly.

What you need before turning it on:

- A working API key in the environment, e.g. `OPENAI_API_KEY` or `ANTHROPIC_API_KEY`.
- A model name that your provider actually accepts. The cached sweep names are evidence from earlier runs, but provider model IDs may differ.
- Enough quota for the stratified sample, up to about `1,000` postings by default. If exposure and automatability use different models, the code makes one call per construct per sampled posting.

The cell writes incremental caches so an interrupted run can resume.


In [51]:
LABEL_PROMPT = """You are a labor-economics researcher assessing how exposed a job is to artificial intelligence, based only on its job posting and description.

For the posting below, estimate two INDEPENDENT probabilities:

1. EXPOSURE - the probability that the role ITSELF involves using, building, or working directly with artificial intelligence, machine learning, or data-science methods. This is about whether AI is integrated into the job's work.

2. AUTOMATABILITY - the probability that the role's CORE tasks are routine and repetitive enough that current computers or AI could perform most of them. This is about whether the job could be done by a machine.

These are independent: a job can be high on one and low on the other.

Return ONLY a JSON object, with no other text:
{
  "p_exposure": <float between 0 and 1>,
  "p_automatable": <float between 0 and 1>,
  "rationale": "<= 20 words explaining the call"
}

Judge strictly from the posting text. Do not use outside knowledge about the company."""


def _extract_json(text):
    text = str(text).strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n?", "", text)
        text = re.sub(r"\n?```$", "", text).strip()
    m = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(m.group(0) if m else text)


def call_openai_chat(model, prompt, api_key=None, timeout=60):
    import requests
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY is not set.")
    r = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={"model": model, "messages": [{"role": "user", "content": prompt}], "temperature": 0},
        timeout=timeout,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


def call_anthropic_messages(model, prompt, api_key=None, timeout=60):
    import requests
    api_key = api_key or os.environ.get("ANTHROPIC_API_KEY", "")
    if not api_key:
        raise RuntimeError("ANTHROPIC_API_KEY is not set.")
    r = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={"x-api-key": api_key, "anthropic-version": "2023-06-01", "Content-Type": "application/json"},
        json={"model": model, "max_tokens": 400, "temperature": 0,
              "messages": [{"role": "user", "content": prompt}]},
        timeout=timeout,
    )
    r.raise_for_status()
    content = r.json()["content"]
    return "".join(block.get("text", "") for block in content if block.get("type") == "text")


def call_llm(model, prompt, provider="openai"):
    if provider == "openai":
        return call_openai_chat(model, prompt)
    if provider == "anthropic":
        return call_anthropic_messages(model, prompt)
    raise ValueError("provider must be 'openai' or 'anthropic'.")


def score_one_posting(text, model, provider, retries=2):
    msg = LABEL_PROMPT + "\n\nJOB POSTING:\n" + str(text)[:MAX_CHARS]
    last = ""
    for attempt in range(retries + 1):
        try:
            raw = call_llm(model, msg, provider=provider)
            d = _extract_json(raw)
            return {"p_exposure": float(d["p_exposure"]),
                    "p_automatable": float(d["p_automatable"]),
                    "rationale": str(d.get("rationale", ""))[:250],
                    "raw_ok": 1}
        except Exception as e:
            last = f"{type(e).__name__}: {e}"
            time.sleep(1.5 * (attempt + 1))
    return {"p_exposure": np.nan, "p_automatable": np.nan, "rationale": f"PARSE_FAIL: {last}", "raw_ok": 0}


In [54]:
# Configure this before the implemented re-score.
# Option B uses a balanced stratified sample instead of the entire 12,487-row corpus.
RESCORE_CONFIG = {
    "exposure": {
        "provider": "openai",
        "model": "gpt-5.4",
    },
    "automatability": {
        "provider": "openai",       # implementation choice: valid + faster/cheaper
        "model": "gpt-5.4-mini",
    },
}

RESCORE_MODE = "stratified_sample"  # "stratified_sample" or "full_corpus"
RESCORE_PER_CELL = 300              # per (year x industry_key); small cells use all rows
RESCORE_LIMIT = None                # optional smoke-test cap after stratification; keep None for Option B


def build_rescore_frame():
    meta_cols = ["job_id", "title_clean", "description_clean", "industry_key", "industry_label", "year", "source_platform"]
    missing = [c for c in meta_cols if c not in postings.columns]
    if missing:
        raise KeyError(f"postings is missing required columns: {missing}. Re-run the setup cells above this re-score section.")

    base = postings.dropna(subset=["year", "industry_key"]).copy()
    if RESCORE_MODE == "stratified_sample":
        sampled_ids = (base.groupby(["year", "industry_key"], group_keys=False)
                       .apply(lambda g: g["job_id"].sample(n=min(len(g), RESCORE_PER_CELL), random_state=42))
                       .astype(str)
                       .tolist())
        order = pd.Series(range(len(sampled_ids)), index=sampled_ids)
        todo = postings[postings["job_id"].astype(str).isin(set(sampled_ids))][meta_cols].copy()
        todo["_sample_order"] = todo["job_id"].astype(str).map(order)
        todo = todo.sort_values("_sample_order").drop(columns="_sample_order").reset_index(drop=True)
    elif RESCORE_MODE == "full_corpus":
        todo = postings[meta_cols].copy().reset_index(drop=True)
    else:
        raise ValueError("RESCORE_MODE must be 'stratified_sample' or 'full_corpus'")

    todo["model_text"] = (todo["title_clean"].fillna("") + " " + todo["description_clean"].fillna(""))
    return todo


def run_construct_rescore(construct, provider, model, limit=None):
    safe_model = str(model).replace("/", "_").replace(" ", "_")
    cache = PROC / f"final_rescore_{construct}_{provider}_{safe_model}.csv"
    todo = build_rescore_frame()

    if cache.exists():
        done = pd.read_csv(cache, dtype={"job_id": str})
        scored = set(done.loc[done.raw_ok.eq(1), "job_id"].astype(str)) if "raw_ok" in done.columns else set(done.job_id.astype(str))
        recs = done.to_dict("records")
        print(f"[{construct}] loaded cache:", len(done), "rows | completed:", len(scored))
    else:
        scored, recs = set(), []

    pending = todo[~todo.job_id.astype(str).isin(scored)]
    if limit is not None:
        pending = pending.head(limit)
    print(f"[{construct}] target rows:", len(todo), "| pending rows this run:", len(pending), "| cache:", cache)

    t0 = time.time()
    for i, (_, row) in enumerate(pending.iterrows(), 1):
        d = score_one_posting(row.model_text, model=model, provider=provider)
        d.update({"job_id": row.job_id, "model": model, "provider": provider, "construct_target": construct})
        recs.append(d)
        if i % 10 == 0:
            pd.DataFrame(recs).to_csv(cache, index=False)
            print(f"[{construct}] {i}/{len(pending)} this run | total cached {len(recs)} | {time.time()-t0:.0f}s")
    pd.DataFrame(recs).to_csv(cache, index=False)
    print(f"[{construct}] wrote", cache, "rows:", len(recs))
    return cache


if FULL_RESCORE:
    sample_frame = build_rescore_frame()
    sample_ids = set(sample_frame["job_id"].astype(str))
    print("re-score mode:", RESCORE_MODE, "| target rows:", len(sample_frame))

    caches = {}
    for construct, cfg in RESCORE_CONFIG.items():
        caches[construct] = run_construct_rescore(
            construct=construct,
            provider=cfg["provider"],
            model=cfg["model"],
            limit=RESCORE_LIMIT,
        )

    # Combine only the selected Option B sample, so the output is not mistaken for a full-corpus score file.
    sample_meta = sample_frame[["job_id", "industry_key", "industry_label", "year", "title_clean", "source_platform"]].copy()
    sample_meta["job_id"] = sample_meta["job_id"].astype(str)

    exp = pd.read_csv(caches["exposure"], dtype={"job_id": str})
    exp = exp[exp["job_id"].astype(str).isin(sample_ids)][["job_id", "p_exposure", "rationale", "raw_ok", "model", "provider"]]
    exp = exp.rename(columns={"p_exposure": "final_exposure_score", "rationale": "exposure_rationale",
                              "raw_ok": "exposure_raw_ok", "model": "exposure_model", "provider": "exposure_provider"})
    auto = pd.read_csv(caches["automatability"], dtype={"job_id": str})
    auto = auto[auto["job_id"].astype(str).isin(sample_ids)][["job_id", "p_automatable", "rationale", "raw_ok", "model", "provider"]]
    auto = auto.rename(columns={"p_automatable": "final_automatable_score", "rationale": "automatable_rationale",
                                "raw_ok": "automatable_raw_ok", "model": "automatable_model", "provider": "automatable_provider"})
    final_scores = (sample_meta
        .merge(exp, on="job_id", how="left")
        .merge(auto, on="job_id", how="left"))
    final_scores["final_exposure"] = pd.Series(pd.NA, index=final_scores.index, dtype="Int64")
    final_scores["final_automatable"] = pd.Series(pd.NA, index=final_scores.index, dtype="Int64")
    exposure_ok = final_scores["final_exposure_score"].notna()
    automatable_ok = final_scores["final_automatable_score"].notna()
    final_scores.loc[exposure_ok, "final_exposure"] = (final_scores.loc[exposure_ok, "final_exposure_score"] >= 0.5).astype(int)
    final_scores.loc[automatable_ok, "final_automatable"] = (final_scores.loc[automatable_ok, "final_automatable_score"] >= 0.5).astype(int)
    out_name = "final_selected_model_scores_stratified_sample.csv" if RESCORE_MODE == "stratified_sample" else "final_selected_model_scores_full_corpus.csv"
    FINAL_OUT = PROC / out_name
    final_scores.to_csv(FINAL_OUT, index=False)
    print("wrote selected-model scores:", FINAL_OUT, "rows:", len(final_scores))
else:
    preview = build_rescore_frame()
    print("FULL_RESCORE is False. Set API key + provider/model, then set FULL_RESCORE=True.")
    print("re-score mode:", RESCORE_MODE, "| target rows:", len(preview), "| RESCORE_LIMIT:", RESCORE_LIMIT)
    print("implemented re-score models:", {"exposure": FINAL_EXPOSURE_MODEL, "automatability": FINAL_AUTOMATABILITY_MODEL, "audit_best_automatability": AUDIT_BEST_AUTOMATABILITY_MODEL})
    print("configured re-score targets:", RESCORE_CONFIG)


re-score mode: stratified_sample | target rows: 2614
[exposure] loaded cache: 2642 rows | completed: 2642
[exposure] target rows: 2614 | pending rows this run: 0 | cache: /Users/PeakViprakasit/QSS45_Final_Project/data/processed/final_rescore_exposure_openai_gpt-5.4.csv
[exposure] wrote /Users/PeakViprakasit/QSS45_Final_Project/data/processed/final_rescore_exposure_openai_gpt-5.4.csv rows: 2642
[automatability] loaded cache: 2628 rows | completed: 2628
[automatability] target rows: 2614 | pending rows this run: 0 | cache: /Users/PeakViprakasit/QSS45_Final_Project/data/processed/final_rescore_automatability_openai_gpt-5.4-mini.csv
[automatability] wrote /Users/PeakViprakasit/QSS45_Final_Project/data/processed/final_rescore_automatability_openai_gpt-5.4-mini.csv rows: 2628
wrote selected-model scores: /Users/PeakViprakasit/QSS45_Final_Project/data/processed/final_selected_model_scores_stratified_sample.csv rows: 2614


## Robustness Appendix: Full Leaderboard

These models are not the main story. They stay here to document that weaker approaches were tested rather than silently ignored.


In [55]:
robustness_leaderboard = all_models.sort_values(["construct", "F1", "AUC"], ascending=[True, False, False]).reset_index(drop=True)
robustness_leaderboard


,construct,model,family,n,pos,AUC,precision,recall,F1,acc,caught
0,automatability,claude-opus-4-8,zero-shot LLM,100,18,0.923,0.625,0.833,0.714,0.88,15/18
1,automatability,gpt-5.4-mini,zero-shot LLM,100,18,0.871,0.484,0.833,0.612,0.81,15/18
2,automatability,mean_gpt_family,ensemble,100,18,0.899,0.522,0.667,0.585,0.83,12/18
3,automatability,Unit8_tfidf_logit,trained supervised,100,18,0.854,0.452,0.778,0.571,0.79,14/18
4,automatability,Unit1_score_logit,trained supervised,100,18,0.852,0.452,0.778,0.571,0.79,14/18
5,automatability,Unit4_random_forest,trained supervised,100,18,0.834,0.643,0.500,0.562,0.86,9/18
6,automatability,gpt-5.4,zero-shot LLM,100,18,0.893,0.556,0.556,0.556,0.84,10/18
7,automatability,mean_all_complete_llm,ensemble,100,18,0.903,0.533,0.444,0.485,0.83,8/18
8,automatability,claude-sonnet-4-6,zero-shot LLM,100,18,0.922,0.857,0.333,0.480,0.87,6/18
9,automatability,median_all_complete_llm,ensemble,100,18,0.896,0.471,0.444,0.457,0.81,8/18


## Methods Note for Write-Up

Use language like this:

> I used a 100-row Claude-drafted validation audit set to compare labelers. Because the audit labels are not independently human-reviewed and contain only five exposure positives, the exact AUC/F1 values should be interpreted as directional rather than definitive. The audit is still useful for model choice: BERT is strong for exposure but weak for automatability, the original full-corpus GPT labeler improves substantially on automatability, and the cached sweep identifies stronger candidates. Course-style supervised models were also trained as a robustness check, but they did not outperform the best zero-shot LLMs on the validation audit.

For final results, be explicit about implementation status:

- If you **do not** run a full-corpus re-score, the existing full-corpus implemented LLM remains `gpt-5.4-mini` on the rows in `gpt_scores.csv` / `ai_scores_panel.csv`, while the final selected models are evaluated on the stratified re-score sample.
- If you **do** run the stratified re-score, restate the 2024→2026 findings as a sample-based robustness check using the new `final_rescore_*.csv` artifact. In the write-up, explain that `claude-opus-4-8` was audit-best for automatability but not available as a working direct API model in the smoke test, so `gpt-5.4-mini` is the implemented automatability model because it is valid, faster/cheaper, and already strong on the validation audit.
